In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
import copy

2025-12-03 12:35:09.584519: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-03 12:35:09.584559: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-03 12:35:09.587568: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-03 12:35:09.621286: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-03 12:35:10.616900: W tensorflow/comp

In [2]:
batch_size = 128
learning_rate = 0.001

In [3]:
@tf.keras.saving.register_keras_serializable()
class MLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense2 = tf.keras.layers.Dense(units=1024, activation=tf.nn.leaky_relu)
        self.dense3 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense4 = tf.keras.layers.Dense(units=256, activation=tf.nn.leaky_relu)
        self.dense5 = tf.keras.layers.Dense(units=8)

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        x = self.dense3(x)
        x = self.dense4(x)
        output = self.dense5(x)
        return output

In [4]:
class ParaServer:
    def __init__(self):
        self.model = MLP()
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    def upload(self, grads):
        self.optimizer.apply_gradients(grads_and_vars=zip(grads, self.model.variables))
        return self.model
    def download(self):
        return self.model
    def initModel(self, x):
        self.model(x)

In [5]:
def valiAll():
    m = ps.download()
    model = copy.deepcopy(m)
    y_v_p = model(X_v)
    va_mse = tf.reduce_mean(tf.square(y_v_p - y_v))
    va_rmse = tf.sqrt(va_mse)
    va_mae = tf.reduce_mean(tf.abs(y_v_p - y_v))
    va_r2 = 1 - tf.reduce_sum(tf.square(y_v_p - y_v)) / tf.reduce_sum(tf.square(y_v - tf.reduce_mean(y_v)))
    print("mse:{} rmse:{} mae:{} r2:{}".format(va_mse, va_rmse, va_mae, va_r2))
    r2sv.append(va_r2.numpy())

In [6]:
class Node:
    def __init__(self, dsName,freq, mu=1e-4):
        self.model = MLP()
        self.freq = freq
        self.mu = mu
        dataset = pd.read_csv(dsName, encoding='utf-8').sample(frac=1).reset_index(drop=True)
        self.X = dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
        self.y = dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)
        self.dataset_train = tf.data.Dataset.from_tensor_slices((self.X, self.y))
        self.dataset_train = self.dataset_train.shuffle(buffer_size=23000)
        self.dataset_train = self.dataset_train.batch(batch_size)
        self.dataset_train = self.dataset_train.prefetch(tf.data.experimental.AUTOTUNE)
    def train(self, num_epochs):
        m = ps.download()
        self.model = copy.deepcopy(m)
        global_weights = [tf.identity(w) for w in self.model.trainable_variables]
        for epoch_index in range(num_epochs):
            for X, y in self.dataset_train:
                with tf.GradientTape() as tape:
                    y_pred = self.model(X)
                    tr_mse = tf.reduce_mean(tf.square(y_pred - y))
                    prox_term = tf.add_n([
                        tf.nn.l2_loss(w - w0)
                        for w, w0 in zip(self.model.trainable_variables, global_weights)
                    ])
                    loss = tr_mse + self.mu * prox_term
                tr_rmse = tf.sqrt(tr_mse)
                tr_mae = tf.reduce_mean(tf.abs(y_pred - y))
                tr_r2 = 1 - tf.reduce_sum(tf.square(y_pred - y)) / tf.reduce_sum(tf.square(y - tf.reduce_mean(y)))
                grads = tape.gradient(loss, self.model.variables)
                m = ps.upload(grads)
                self.model = copy.deepcopy(m)
                # if epoch_index in np.arange(0, num_epochs, 25).tolist() or epoch_index == num_epochs - 1:
            if True:
                print("node:{} epoch:{}".format(self.freq, epoch_index))
                print("train mse:{} rmse:{} mae:{} r2:{}".format(tr_mse, tr_rmse, tr_mae, tr_r2))
                r2s.append(tr_r2.numpy())
                valiAll()

In [8]:
r2s = []
r2sv = []

In [9]:
test_dataset = pd.read_csv("Test.csv", encoding='utf-8').sample(frac=1).reset_index(drop=True)
X_v = test_dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
y_v = test_dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)

In [10]:
ps = ParaServer()
ps.initModel(X_v)

2025-12-03 12:35:25.440457: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9610 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:17:00.0, compute capability: 7.5
2025-12-03 12:35:25.441715: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 9554 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:65:00.0, compute capability: 7.5


In [11]:
nodeList = [Node('./24Train.csv', 2.4), Node('./25Train.csv', 2.5), Node('./26Train.csv', 2.6)]

In [ ]:
nodeList[0].train(150)
nodeList[1].train(150)
nodeList[2].train(150)
nodeList[0].train(150)
nodeList[1].train(150)
nodeList[2].train(150)

2025-12-03 12:35:28.976116: I external/local_xla/xla/service/service.cc:168] XLA service 0x58f2cff6d4d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-12-03 12:35:28.976136: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5
2025-12-03 12:35:28.976143: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (1): NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5
2025-12-03 12:35:28.981627: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-12-03 12:35:29.003101: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8902
I0000 00:00:1764765329.083617   23757 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


node:2.4 epoch:0
train mse:0.0826907679438591 rmse:0.28756001591682434 mae:0.22861996293067932 r2:0.31809180974960327
mse:0.08816134929656982 rmse:0.2969197630882263 mae:0.24067996442317963 r2:0.2702106833457947
node:2.4 epoch:1
train mse:0.0717984288930893 rmse:0.2679522931575775 mae:0.2136823832988739 r2:0.4094281792640686
mse:0.0850096195936203 rmse:0.29156407713890076 mae:0.2354930341243744 r2:0.29630035161972046
node:2.4 epoch:2
train mse:0.07184522598981857 rmse:0.2680395841598511 mae:0.21409575641155243 r2:0.40392476320266724
mse:0.07634063810110092 rmse:0.27629807591438293 mae:0.22250832617282867 r2:0.3680611252784729
node:2.4 epoch:3
train mse:0.07314132153987885 rmse:0.2704465091228485 mae:0.2119099348783493 r2:0.3895276188850403
mse:0.07440868020057678 rmse:0.27277952432632446 mae:0.21836967766284943 r2:0.38405364751815796
node:2.4 epoch:4
train mse:0.06720104813575745 rmse:0.25923165678977966 mae:0.20672594010829926 r2:0.4444234371185303
mse:0.07215374708175659 rmse:0.26861

In [21]:
for i in r2sv:
    print(i)

0.05343145
0.21146125
0.2544238
0.2703812
0.2867133
0.30649197
0.31547016
0.3224408
0.3291536
0.3452314
0.34077
0.3562141
0.354967
0.35245204
0.32595384
0.40231603
0.40158945
0.40962702
0.42272305
0.4230671
0.4107265
0.41005546
0.436269
0.44843858
0.43839198
0.43388087
0.45969415
0.45770818
0.4541483
0.4664585
0.47532052
0.46199465
0.47981328
0.47990716
0.49459165
0.4690683
0.45069534
0.48096782
0.49847054
0.46773773
0.4992084
0.47926748
0.49972522
0.48676974
0.4963382
0.48902565
0.5175091
0.50038993
0.46291137
0.5333949
0.53417766
0.48639023
0.55056655
0.5456645
0.53572404
0.5499059
0.5739113
0.5726086
0.47727573
0.5278763
0.49325925
0.58120906
0.47513604
0.5654805
0.5336244
0.4959963
0.5241405
0.5895729
0.45611674
0.37396538
0.50237
0.46507394
0.513464
0.49617326
0.5751781
0.5460856
0.4678169
0.5107386
0.5489664
0.61125076
0.45940083
0.43182415
0.49852914
0.52809775
0.47194016
0.5567453
0.496157
0.3778242
0.471923
0.41169584
0.5299705
0.43949437
0.41430855
0.38863832
0.47739798
0.538